In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/mock")

preferences = pd.read_csv(
    DATA_DIR / "user_preferences.csv"
)

display(preferences)

,user_id,aspect,importance,confidence,max_budget
0,U001,battery,5,0.90,170000
1,U001,camera,3,0.70,170000
2,U001,performance,4,0.85,170000
3,U001,display,3,0.80,170000
4,U001,durability,2,0.75,170000
5,U001,price,4,0.90,170000


In [2]:
pref = preferences.copy()

pref["nominal_weight"] = (
    pref["importance"] /
    pref["importance"].sum()
)

display(
    pref[
        [
            "aspect",
            "importance",
            "confidence",
            "nominal_weight"
        ]
    ]
)

print(
    "Weight sum:",
    pref["nominal_weight"].sum()
)

,aspect,importance,confidence,nominal_weight
0,battery,5,0.90,0.238095
1,camera,3,0.70,0.142857
2,performance,4,0.85,0.190476
3,display,3,0.80,0.142857
4,durability,2,0.75,0.095238
5,price,4,0.90,0.190476


Weight sum: 0.9999999999999999


In [3]:
def sample_preferences(
    preferences,
    n_samples=500,
    uncertainty_scale=1.0,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    base = preferences.copy()

    base["nominal_weight"] = (
        base["importance"] /
        base["importance"].sum()
    )

    samples = []

    for simulation_id in range(n_samples):

        sampled_raw = []

        for _, row in base.iterrows():

            sigma = (
                uncertainty_scale
                * (1 - row["confidence"])
            )

            noise = rng.normal(
                loc=0.0,
                scale=sigma
            )

            perturbed_weight = (
                row["nominal_weight"]
                * np.exp(noise)
            )

            sampled_raw.append(
                perturbed_weight
            )

        sampled_raw = np.array(sampled_raw)

        sampled_weights = (
            sampled_raw /
            sampled_raw.sum()
        )

        for i, (_, row) in enumerate(base.iterrows()):
            samples.append({
                "simulation_id": simulation_id,
                "aspect": row["aspect"],
                "importance": row["importance"],
                "confidence": row["confidence"],
                "nominal_weight": row["nominal_weight"],
                "sampled_weight": sampled_weights[i]
            })

    return pd.DataFrame(samples)

In [4]:
preference_samples = sample_preferences(
    preferences,
    n_samples=500,
    uncertainty_scale=1.0,
    random_state=42
)

print(
    "Generated rows:",
    len(preference_samples)
)

display(
    preference_samples.head(12)
)

Generated rows: 3000


,simulation_id,aspect,importance,confidence,nominal_weight,sampled_weight
0,0,battery,5,0.90,0.238095,0.255338
1,0,camera,3,0.70,0.142857,0.108776
2,0,performance,4,0.85,0.190476,0.221748
3,0,display,3,0.80,0.142857,0.179361
4,0,durability,2,0.75,0.095238,0.060829
5,0,price,4,0.90,0.190476,0.173948
6,1,battery,5,0.90,0.238095,0.239703
7,1,camera,3,0.70,0.142857,0.129143
8,1,performance,4,0.85,0.190476,0.188850
9,1,display,3,0.80,0.142857,0.119723


In [5]:
weight_sums = (
    preference_samples
    .groupby("simulation_id")
    ["sampled_weight"]
    .sum()
)

print(weight_sums.describe())

assert np.allclose(
    weight_sums.values,
    1.0
)

print("WEIGHT SUM TEST: PASSED")

count    5.000000e+02
mean     1.000000e+00
std      8.013950e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: sampled_weight, dtype: float64
WEIGHT SUM TEST: PASSED


In [6]:
uncertainty_summary = (
    preference_samples
    .groupby("aspect")
    .agg(
        importance=("importance", "first"),
        confidence=("confidence", "first"),
        nominal_weight=("nominal_weight", "first"),
        mean_sampled_weight=("sampled_weight", "mean"),
        std_sampled_weight=("sampled_weight", "std"),
        min_sampled_weight=("sampled_weight", "min"),
        max_sampled_weight=("sampled_weight", "max")
    )
    .reset_index()
)

display(uncertainty_summary)

,aspect,importance,confidence,nominal_weight,mean_sampled_weight,std_sampled_weight,min_sampled_weight,max_sampled_weight
0,battery,5,0.90,0.238095,0.236070,0.025154,0.173250,0.333511
1,camera,3,0.70,0.142857,0.144709,0.039002,0.054611,0.302898
2,display,3,0.80,0.142857,0.144755,0.027240,0.083373,0.230711
3,durability,2,0.75,0.095238,0.096353,0.021791,0.046398,0.172624
4,performance,4,0.85,0.190476,0.189047,0.025894,0.122617,0.276932
5,price,4,0.90,0.190476,0.189065,0.019502,0.131471,0.259497


In [8]:
test_preferences = pd.DataFrame({
    "user_id": ["TEST", "TEST", "TEST"],
    "aspect": [
        "high_confidence",
        "low_confidence",
        "anchor"
    ],
    "importance": [3, 3, 3],
    "confidence": [0.95, 0.50, 1.00],
    "max_budget": [170000, 170000, 170000]
})

test_samples = sample_preferences(
    test_preferences,
    n_samples=2000,
    random_state=42
)

test_summary = (
    test_samples
    .groupby("aspect")
    ["sampled_weight"]
    .std()
)

display(test_summary)

aspect
anchor             0.054342
high_confidence    0.054812
low_confidence     0.107909
Name: sampled_weight, dtype: float64

In [9]:
high_std = test_summary["high_confidence"]
low_std = test_summary["low_confidence"]

assert low_std > high_std

print("CONFIDENCE EFFECT TEST: PASSED")

print("High-confidence variation:", high_std)
print("Low-confidence variation:", low_std)

CONFIDENCE EFFECT TEST: PASSED
High-confidence variation: 0.05481194975774407
Low-confidence variation: 0.10790864162447066


In [10]:
products = pd.read_csv(DATA_DIR / "products.csv")
c1 = pd.read_csv(DATA_DIR / "c1_product_aspects.csv")
c3 = pd.read_csv(DATA_DIR / "c3_temporal_features.csv")

features = c1.merge(
    c3,
    on=["product_id", "aspect"],
    how="inner"
)

features["nominal_aspect_score"] = (
    features["aspect_quality"]
    + features["recent_quality"]
) / 2

display(
    features[
        [
            "product_id",
            "aspect",
            "nominal_aspect_score"
        ]
    ].head(10)
)

,product_id,aspect,nominal_aspect_score
0,P001,battery,0.890
1,P001,camera,0.735
2,P001,performance,0.905
3,P001,display,0.815
4,P001,durability,0.785
5,P001,price,0.695
6,P002,battery,0.735
7,P002,camera,0.900
8,P002,performance,0.830
9,P002,display,0.855


In [11]:
simulation_features = features.merge(
    preference_samples[
        [
            "simulation_id",
            "aspect",
            "sampled_weight"
        ]
    ],
    on="aspect",
    how="inner"
)

simulation_features["weighted_contribution"] = (
    simulation_features["nominal_aspect_score"]
    * simulation_features["sampled_weight"]
)

simulation_scores = (
    simulation_features
    .groupby(
        ["simulation_id", "product_id"],
        as_index=False
    )
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution":
            "personalized_score"
        }
    )
)

simulation_scores = simulation_scores.merge(
    products,
    on="product_id",
    how="left"
)

max_budget = preferences["max_budget"].iloc[0]

simulation_scores = simulation_scores[
    simulation_scores["price"] <= max_budget
].copy()

simulation_scores = (
    simulation_scores
    .sort_values(
        ["simulation_id", "personalized_score"],
        ascending=[True, False]
    )
)

simulation_scores["rank"] = (
    simulation_scores
    .groupby("simulation_id")
    .cumcount() + 1
)

display(simulation_scores.head(12))

,simulation_id,product_id,personalized_score,product_name,category,price,rank
0,0,P001,0.822707,Phone A,smartphone,165000,1
4,0,P005,0.821986,Phone E,smartphone,160000,2
1,0,P002,0.811235,Phone B,smartphone,150000,3
3,0,P004,0.776217,Phone D,smartphone,140000,4
9,1,P005,0.825542,Phone E,smartphone,160000,1
5,1,P001,0.811548,Phone A,smartphone,165000,2
6,1,P002,0.807780,Phone B,smartphone,150000,3
8,1,P004,0.781897,Phone D,smartphone,140000,4
14,2,P005,0.823100,Phone E,smartphone,160000,1
11,2,P002,0.814817,Phone B,smartphone,150000,2


In [12]:
print(
    "Number of simulations:",
    simulation_scores["simulation_id"].nunique()
)

print(
    "Ranking rows:",
    len(simulation_scores)
)

Number of simulations: 500
Ranking rows: 2000


In [13]:
top1_results = (
    simulation_scores[
        simulation_scores["rank"] == 1
    ]
)

top1_summary = (
    top1_results
    .groupby(
        ["product_id", "product_name"]
    )
    .size()
    .reset_index(
        name="top1_count"
    )
)

top1_summary["top1_probability"] = (
    top1_summary["top1_count"]
    / top1_results["simulation_id"].nunique()
)

top1_summary = top1_summary.sort_values(
    "top1_count",
    ascending=False
)

display(top1_summary)

,product_id,product_name,top1_count,top1_probability
2,P005,Phone E,457,0.914
1,P002,Phone B,37,0.074
0,P001,Phone A,6,0.012


In [14]:
top1_stability = (
    top1_summary["top1_count"].max()
    / 500
)

print(
    "Preference-only Top-1 Stability:",
    round(top1_stability, 4)
)

Preference-only Top-1 Stability: 0.914


In [15]:
unique_winners = (
    top1_results["product_id"].nunique()
)

print(
    "Number of different Rank-1 winners:",
    unique_winners
)

if unique_winners == 1:
    print(
        "Ranking is highly stable under current "
        "preference uncertainty."
    )
else:
    print(
        "Ranking changes under preference uncertainty."
    )

Number of different Rank-1 winners: 3
Ranking changes under preference uncertainty.


In [16]:
assert (
    simulation_scores["simulation_id"].nunique()
    == 500
)

assert (
    top1_results.groupby("simulation_id").size()
    == 1
).all()

assert np.isclose(
    top1_summary["top1_probability"].sum(),
    1.0
)

print(
    "PREFERENCE-UNCERTAINTY RANKING TEST: PASSED"
)

PREFERENCE-UNCERTAINTY RANKING TEST: PASSED


In [17]:
# Nominal baseline ranking without preference perturbation

nominal_pref = preferences.copy()

nominal_pref["nominal_weight"] = (
    nominal_pref["importance"]
    / nominal_pref["importance"].sum()
)

nominal_features = features.merge(
    nominal_pref[["aspect", "nominal_weight"]],
    on="aspect",
    how="inner"
)

nominal_features["weighted_contribution"] = (
    nominal_features["nominal_aspect_score"]
    * nominal_features["nominal_weight"]
)

nominal_scores = (
    nominal_features
    .groupby("product_id", as_index=False)
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution": "personalized_score"
        }
    )
)

nominal_scores = nominal_scores.merge(
    products,
    on="product_id",
    how="left"
)

nominal_scores = nominal_scores[
    nominal_scores["price"] <= max_budget
]

nominal_scores = nominal_scores.sort_values(
    "personalized_score",
    ascending=False
)

nominal_winner = nominal_scores.iloc[0]["product_id"]

print("Nominal winner:", nominal_winner)

Nominal winner: P005


In [18]:
nominal_winner_count = (
    top1_results["product_id"] == nominal_winner
).sum()

preference_top1_stability = (
    nominal_winner_count
    / top1_results["simulation_id"].nunique()
)

print(
    "Preference-only Top-1 Stability:",
    round(preference_top1_stability, 4)
)

print(
    "Nominal winner retained in:",
    nominal_winner_count,
    "out of",
    top1_results["simulation_id"].nunique(),
    "simulations"
)

Preference-only Top-1 Stability: 0.914
Nominal winner retained in: 457 out of 500 simulations
